# 03 — Construction des agrégats OLAP

## Plan
1. Imports et chargement
2. Définition des KPI et dimensions
3. Agrégation par département
4. Agrégation par milieu
5. Agrégation par secteur d'activité
6. Agrégation par niveau d'éducation
7. Agrégation par niveau d'inclusion financière
8. Agrégation par genre
9. Agrégation par tranche d'âge
10. Agrégation croisée département × milieu
11. Agrégation croisée secteur × milieu
12. Table de ciblage microcrédit (score de priorité par département)
13. Export de tous les fichiers


## 1. Imports et chargement

In [47]:
import pandas as pd
import numpy as np
import os

# ── Chemins ──────────────────────────────────────────────────────────────────
FILE_RAW    = "C:\\Users\\DELL\\Desktop\\Formation Data-science_FRST_FDS\\Projet_Capstone\\Segmentation_Inclusion_Financiere_Groupe5\\Donnees\\Donnees_Nettoyees\\FinScope_Haiti_2018_Données nettoyées.xlsx"
SHEET       = "Données_Nettoyées"
DIR_OUT     = "Outputs\\Tables"
os.makedirs(DIR_OUT, exist_ok=True)

# ── Chargement et extraction population cible ────────────────────────────────
df_brut = pd.read_excel(FILE_RAW, sheet_name=SHEET)
df = df_brut[df_brut["Jamais_Eu_Credit_Formel"] == "Oui"].copy()
df.reset_index(drop=True, inplace=True)

print(f"Population cible : {len(df):,} individus × {df.shape[1]} variables")
df.head(3)


Population cible : 4,016 individus × 34 variables


,ID_Individu,Departement,Milieu,Genre,Tranche_Age,Age,Niveau_Education,Secteur_Activite,Revenu_Mensuel_HTG,Possede_Telephone,...,Epargne_Formelle,Membre_Tontine_Sol,Mobile_Money,Planification_Financiere,Controle_Depenses,Manque_Argent_Frequent,Raison_Principale_Manque_Argent,Besoin_Education_Financiere,Jamais_Eu_Credit_Formel,Poids_Sondage
0,HTI2018_0001,Grand-Anse,Rural,Homme,35-44,36,Secondaire,Artisanat/Production,3514,Non,...,Non,Non,Non,Pas souvent,Contrôle avec d'autres,Oui,Médicaments/Santé,Comment obtenir un prêt,Oui,0.8367
1,HTI2018_0002,Aire Métropolitaine,Urbain,Femme,25-34,29,Primaire,Artisanat/Production,7021,Oui,...,Oui,Oui,Non,Souvent,Ne sait pas,Oui,Pas assez de revenu,Comment budgétiser,Oui,1.2801
2,HTI2018_0003,Nord-Est,Urbain,Homme,45-60,55,Non scolarisé,Commerce informel,4151,Oui,...,Non,Oui,Non,Souvent,Contrôle total,Oui,Pas assez de revenu,Ne sait pas/Aucun besoin,Oui,0.9448


## 2. Définition des KPI et fonction d'agrégation

In [48]:
# ── Fonction utilitaire : agrégation multi-KPI ───────────────────────────────
def agreger(dataframe, groupby_cols):
    """
    Calcule les 8 KPI principaux pour un groupe donné.
    Retourne un DataFrame agrégé prêt pour le dashboard.
    """
    agg = dataframe.groupby(groupby_cols).agg(
        n_individus          = ("ID_Individu",             "count"),
        revenu_median        = ("Revenu_Mensuel_HTG",      "median"),
        revenu_moyen         = ("Revenu_Mensuel_HTG",      "mean"),
        revenu_std           = ("Revenu_Mensuel_HTG",      "std"),
    ).reset_index()

    # KPI binaires — calculés séparément puis fusionnés
    kpis_bin = {
        "taux_exclusion_financiere" : ("Niveau_Inclusion_Financiere", "Exclus financièrement"),
        "taux_bancarise"            : ("Niveau_Inclusion_Financiere", "Bancarisé"),
        "taux_cin"                  : ("CIN_Carte_Identite",          "Oui"),
        "taux_tontine"              : ("Membre_Tontine_Sol",          "Oui"),
        "taux_mobile_money"         : ("Mobile_Money",                "Oui"),
        "taux_epargne_formelle"     : ("Epargne_Formelle",            "Oui"),
        "taux_credit_informel"      : ("Credit_Informel",             "Oui"),
        "taux_manque_argent"        : ("Manque_Argent_Frequent",      "Oui"),
        "taux_telephone"            : ("Possede_Telephone",           "Oui"),
        "taux_nif"                  : ("NIF_Matricule_Fiscal",        "Oui"),
        "taux_compte_bancaire"      : ("Compte_Bancaire",             "Oui"),
        "taux_planif_jamais"        : ("Planification_Financiere",    "Jamais"),
    }

    for col_name, (var, val) in kpis_bin.items():
        tmp = dataframe.copy()
        tmp["_flag"] = (tmp[var] == val).astype(int)
        taux = tmp.groupby(groupby_cols)["_flag"].mean().reset_index()
        taux.columns = list(groupby_cols) + [col_name]
        agg = agg.merge(taux, on=groupby_cols, how="left")

    # Arrondir
    for col in agg.select_dtypes("float").columns:
        agg[col] = agg[col].round(4)

    return agg

print(" Fonction d'agrégation définie — 12 KPI calculés par groupe")
print("   KPI : n_individus, revenu_médian, revenu_moyen, revenu_std,")
print("         exclusion financière, bancarisé, CIN, tontine, mobile money,")
print("         épargne formelle, crédit informel, manque d'argent,")
print("         téléphone, NIF, compte bancaire, planif jamais")


 Fonction d'agrégation définie — 12 KPI calculés par groupe
   KPI : n_individus, revenu_médian, revenu_moyen, revenu_std,
         exclusion financière, bancarisé, CIN, tontine, mobile money,
         épargne formelle, crédit informel, manque d'argent,
         téléphone, NIF, compte bancaire, planif jamais


## 3. Agrégation par département

In [49]:
agg_dept = agreger(df, ["Departement"])

# Ordre géographique pour les graphiques
ordre_dept = df.groupby("Departement")["Revenu_Mensuel_HTG"].median()               .sort_values(ascending=False).index.tolist()
agg_dept["ordre"] = agg_dept["Departement"].map({d: i for i, d in enumerate(ordre_dept)})
agg_dept = agg_dept.sort_values("ordre").drop(columns="ordre").reset_index(drop=True)

print(f"Agrégat département : {agg_dept.shape}")
agg_dept[["Departement","n_individus","revenu_median","taux_exclusion_financiere",
           "taux_cin","taux_tontine","taux_mobile_money","taux_manque_argent"]]


Agrégat département : (10, 17)


,Departement,n_individus,revenu_median,taux_exclusion_financiere,taux_cin,taux_tontine,taux_mobile_money,taux_manque_argent
0,Aire Métropolitaine,1461,3373.0,0.3176,0.6927,0.4045,0.1937,0.6188
1,Nord,393,3019.0,0.5394,0.6336,0.4402,0.0789,0.6158
2,Sud,367,2756.0,0.5586,0.6785,0.4169,0.0708,0.6158
3,Nord-Est,181,2694.0,0.5801,0.6133,0.4144,0.0773,0.5856
4,Ouest (reste),314,2508.5,0.6115,0.6242,0.4140,0.1083,0.6146
5,Artibonite,503,2431.0,0.5666,0.6521,0.3757,0.0915,0.6441
6,Centre,281,2161.0,0.5694,0.6299,0.4164,0.0534,0.6690
7,Nippes,179,2097.0,0.6201,0.5922,0.4190,0.0782,0.6704
8,Grand-Anse,201,1894.0,0.5871,0.5821,0.3682,0.0647,0.6816
9,Nord-Ouest,136,1540.0,0.6176,0.5588,0.3824,0.0441,0.7500


## 4. Agrégation par milieu

In [50]:
agg_milieu = agreger(df, ["Milieu"])
print(f"Agrégat milieu : {agg_milieu.shape}")
agg_milieu


Agrégat milieu : (2, 17)


,Milieu,n_individus,revenu_median,revenu_moyen,revenu_std,taux_exclusion_financiere,taux_bancarise,taux_cin,taux_tontine,taux_mobile_money,taux_epargne_formelle,taux_credit_informel,taux_manque_argent,taux_telephone,taux_nif,taux_compte_bancaire,taux_planif_jamais
0,Rural,1176,1540.0,2291.8129,1940.7746,0.5697,0.0927,0.5451,0.4082,0.0587,0.0825,0.0697,0.6845,0.3614,0.2551,0.0927,0.3988
1,Urbain,2840,3380.0,4126.8296,3773.9816,0.4458,0.1701,0.6972,0.4046,0.1454,0.1468,0.0697,0.6116,0.7127,0.2979,0.1701,0.3070


## 5. Agrégation par secteur d'activité

In [51]:
agg_secteur = agreger(df, ["Secteur_Activite"])

# Ordre par revenu médian décroissant
agg_secteur = agg_secteur.sort_values("revenu_median", ascending=False).reset_index(drop=True)
print(f"Agrégat secteur : {agg_secteur.shape}")
agg_secteur[["Secteur_Activite","n_individus","revenu_median",
              "taux_tontine","taux_mobile_money","taux_epargne_formelle"]]


Agrégat secteur : (6, 17)


,Secteur_Activite,n_individus,revenu_median,taux_tontine,taux_mobile_money,taux_epargne_formelle
0,Emploi salarié formel,307,11207.0,0.3257,0.1824,0.3160
1,Services/Transport,529,5215.0,0.3762,0.1418,0.1399
2,Commerce informel,1026,4151.0,0.4903,0.1228,0.1365
3,Artisanat/Production,439,3380.0,0.3235,0.1093,0.1071
4,Agriculture/Élevage,1116,1592.0,0.4014,0.0932,0.0842
5,Sans activité/Ménage,599,631.0,0.3957,0.1219,0.1035


## 6. Agrégation par niveau d'éducation

In [52]:
# Encodage ordinal pour tri logique
ordre_educ = ["Non scolarisé", "Primaire", "Secondaire", "Supérieur/Professionnel"]
agg_educ = agreger(df, ["Niveau_Education"])
agg_educ["ordre"] = agg_educ["Niveau_Education"].map({v: i for i, v in enumerate(ordre_educ)})
agg_educ = agg_educ.sort_values("ordre").drop(columns="ordre").reset_index(drop=True)

print(f"Agrégat éducation : {agg_educ.shape}")
agg_educ[["Niveau_Education","n_individus","revenu_median","taux_cin","taux_mobile_money"]]


Agrégat éducation : (4, 17)


,Niveau_Education,n_individus,revenu_median,taux_cin,taux_mobile_money
0,Non scolarisé,906,2093.0,0.5166,0.0883
1,Primaire,1076,2315.5,0.6589,0.0929
2,Secondaire,1777,3173.0,0.7107,0.1491
3,Supérieur/Professionnel,257,3380.0,0.7043,0.1440


## 7. Agrégation par niveau d'inclusion financière

In [53]:
ordre_incl = ["Exclus financièrement", "Informel seulement",
               "Autre formel (non-bancaire)", "Bancarisé"]
agg_inclusion = agreger(df, ["Niveau_Inclusion_Financiere"])
agg_inclusion["ordre"] = agg_inclusion["Niveau_Inclusion_Financiere"].map(
    {v: i for i, v in enumerate(ordre_incl)})
agg_inclusion = agg_inclusion.sort_values("ordre").drop(columns="ordre").reset_index(drop=True)

print(f"Agrégat inclusion : {agg_inclusion.shape}")
agg_inclusion[["Niveau_Inclusion_Financiere","n_individus","revenu_median","taux_tontine"]]


Agrégat inclusion : (4, 17)


,Niveau_Inclusion_Financiere,n_individus,revenu_median,taux_tontine
0,Exclus financièrement,1936,2144.0,0.4298
1,Informel seulement,392,2756.0,0.3954
2,Autre formel (non-bancaire),1096,3166.5,0.3887
3,Bancarisé,592,3934.0,0.3649


## 8. Agrégation par genre

In [54]:
agg_genre = agreger(df, ["Genre"])
print(f"Agrégat genre : {agg_genre.shape}")
agg_genre


Agrégat genre : (2, 17)


,Genre,n_individus,revenu_median,revenu_moyen,revenu_std,taux_exclusion_financiere,taux_bancarise,taux_cin,taux_tontine,taux_mobile_money,taux_epargne_formelle,taux_credit_informel,taux_manque_argent,taux_telephone,taux_nif,taux_compte_bancaire,taux_planif_jamais
0,Femme,2047,2756.0,3559.2931,3397.9330,0.4846,0.1417,0.6532,0.4558,0.1163,0.1246,0.0694,0.6282,0.6043,0.2755,0.1417,0.3351
1,Homme,1969,2737.0,3620.8710,3494.6963,0.4794,0.1534,0.6521,0.3535,0.1239,0.1315,0.0701,0.6379,0.6155,0.2956,0.1534,0.3327


## 9. Agrégation par tranche d'âge

In [55]:
ordre_age = ["15-17", "18-24", "25-34", "35-44", "45-60", "60+"]
agg_age = agreger(df, ["Tranche_Age"])
agg_age["ordre"] = agg_age["Tranche_Age"].map({v: i for i, v in enumerate(ordre_age)})
agg_age = agg_age.sort_values("ordre").drop(columns="ordre").reset_index(drop=True)

print(f"Agrégat tranche d'âge : {agg_age.shape}")
agg_age[["Tranche_Age","n_individus","revenu_median","taux_tontine","taux_mobile_money"]]


Agrégat tranche d'âge : (6, 17)


,Tranche_Age,n_individus,revenu_median,taux_tontine,taux_mobile_money
0,15-17,374,2740.5,0.4037,0.1310
1,18-24,721,2631.0,0.4300,0.1221
2,25-34,987,2838.0,0.3850,0.1145
3,35-44,863,2756.0,0.4021,0.1240
4,45-60,699,2535.0,0.4206,0.1202
5,60+,372,2751.5,0.3952,0.1102


## 10. Agrégation croisée département × milieu

In [56]:
agg_dept_milieu = agreger(df, ["Departement", "Milieu"])
print(f"Agrégat département × milieu : {agg_dept_milieu.shape}")
agg_dept_milieu.head(10)


Agrégat département × milieu : (20, 18)


,Departement,Milieu,n_individus,revenu_median,revenu_moyen,revenu_std,taux_exclusion_financiere,taux_bancarise,taux_cin,taux_tontine,taux_mobile_money,taux_epargne_formelle,taux_credit_informel,taux_manque_argent,taux_telephone,taux_nif,taux_compte_bancaire,taux_planif_jamais
0,Aire Métropolitaine,Rural,303,1633.0,2393.8911,1941.2462,0.4323,0.1749,0.5809,0.4125,0.1023,0.1221,0.0660,0.7063,0.3762,0.2376,0.1749,0.3795
1,Aire Métropolitaine,Urbain,1158,3963.5,4649.1675,4255.6125,0.2876,0.2478,0.7219,0.4024,0.2176,0.1943,0.0699,0.5959,0.7383,0.3247,0.2478,0.2893
2,Artibonite,Rural,101,1540.0,2516.1089,2712.1197,0.6535,0.0693,0.5743,0.4059,0.0396,0.0693,0.0594,0.6832,0.2970,0.2475,0.0693,0.3564
3,Artibonite,Urbain,402,2667.5,3407.2214,2980.4367,0.5448,0.0970,0.6716,0.3682,0.1045,0.1219,0.0672,0.6343,0.6741,0.2587,0.0970,0.3308
4,Centre,Rural,172,1747.0,2484.2151,1972.1947,0.5756,0.0581,0.5640,0.4186,0.0407,0.0581,0.0291,0.6628,0.4070,0.2849,0.0581,0.4593
5,Centre,Urbain,109,2732.0,3341.4037,2884.9797,0.5596,0.1376,0.7339,0.4128,0.0734,0.1101,0.0459,0.6789,0.6881,0.2844,0.1376,0.3578
6,Grand-Anse,Rural,130,1650.5,2266.8231,1890.6786,0.5692,0.1000,0.5615,0.3462,0.0385,0.0923,0.0923,0.6923,0.2846,0.2769,0.1000,0.3846
7,Grand-Anse,Urbain,71,2102.0,3483.4225,3816.7736,0.6197,0.1127,0.6197,0.4085,0.1127,0.1268,0.0563,0.6620,0.7183,0.2254,0.1127,0.2535
8,Nippes,Rural,109,1769.0,2324.2752,1922.8242,0.6330,0.0550,0.5229,0.4128,0.0459,0.0734,0.0826,0.6881,0.4037,0.2110,0.0550,0.3486
9,Nippes,Urbain,70,2812.5,3254.2571,2678.8010,0.6000,0.1571,0.7000,0.4286,0.1286,0.1714,0.0429,0.6429,0.8000,0.2857,0.1571,0.3000


## 11. Agrégation croisée secteur × milieu

In [57]:
agg_secteur_milieu = agreger(df, ["Secteur_Activite", "Milieu"])
print(f"Agrégat secteur × milieu : {agg_secteur_milieu.shape}")
agg_secteur_milieu.head(8)


Agrégat secteur × milieu : (12, 18)


,Secteur_Activite,Milieu,n_individus,revenu_median,revenu_moyen,revenu_std,taux_exclusion_financiere,taux_bancarise,taux_cin,taux_tontine,taux_mobile_money,taux_epargne_formelle,taux_credit_informel,taux_manque_argent,taux_telephone,taux_nif,taux_compte_bancaire,taux_planif_jamais
0,Agriculture/Élevage,Rural,631,1540.0,1689.5436,960.7803,0.5880,0.0935,0.5357,0.4025,0.0650,0.0761,0.0618,0.7132,0.3756,0.2599,0.0935,0.4041
1,Agriculture/Élevage,Urbain,485,1998.0,2116.4887,1093.5167,0.6124,0.0866,0.6701,0.4000,0.1299,0.0948,0.0784,0.6474,0.7113,0.2351,0.0866,0.3031
2,Artisanat/Production,Rural,101,2756.0,2853.9208,1480.1887,0.5644,0.0594,0.5446,0.2376,0.0396,0.0693,0.0891,0.6337,0.3663,0.2871,0.0594,0.4455
3,Artisanat/Production,Urbain,338,3380.0,3551.9734,1903.6203,0.5148,0.1361,0.7041,0.3491,0.1302,0.1183,0.0621,0.6568,0.7071,0.2870,0.1361,0.3077
4,Commerce informel,Rural,217,3355.0,3437.2581,1687.9971,0.5069,0.0968,0.5438,0.5115,0.0507,0.1106,0.0829,0.6313,0.3410,0.2304,0.0968,0.3825
5,Commerce informel,Urbain,809,4151.0,4186.7417,2208.0091,0.3807,0.1780,0.7021,0.4845,0.1422,0.1434,0.0729,0.6082,0.6823,0.2645,0.1780,0.3066
6,Emploi salarié formel,Rural,18,8992.0,9357.8889,4918.9275,0.0556,0.5000,0.6667,0.3889,0.1111,0.2778,0.0000,0.4444,0.5556,0.6111,0.5000,0.2778
7,Emploi salarié formel,Urbain,289,11207.0,11320.5017,5106.8590,0.1176,0.4152,0.7197,0.3218,0.1869,0.3183,0.0830,0.4360,0.7474,0.6228,0.4152,0.3183


## 12. Agrégation croisée secteur × genre (comportements financiers)

In [58]:
agg_secteur_genre = agreger(df, ["Secteur_Activite", "Genre"])
print(f"Agrégat secteur × genre : {agg_secteur_genre.shape}")
agg_secteur_genre.head(6)


Agrégat secteur × genre : (12, 18)


,Secteur_Activite,Genre,n_individus,revenu_median,revenu_moyen,revenu_std,taux_exclusion_financiere,taux_bancarise,taux_cin,taux_tontine,taux_mobile_money,taux_epargne_formelle,taux_credit_informel,taux_manque_argent,taux_telephone,taux_nif,taux_compte_bancaire,taux_planif_jamais
0,Agriculture/Élevage,Femme,569,1604.0,1924.8366,1055.7996,0.6046,0.0808,0.6046,0.4569,0.0879,0.0791,0.0721,0.6696,0.5079,0.2408,0.0808,0.3673
1,Agriculture/Élevage,Homme,547,1573.0,1823.3400,1025.5576,0.5923,0.1005,0.5832,0.3437,0.0987,0.0896,0.0658,0.7002,0.5356,0.2578,0.1005,0.3528
2,Artisanat/Production,Femme,208,3380.0,3319.3462,1850.6540,0.4856,0.1298,0.6923,0.3606,0.0865,0.1250,0.0673,0.6683,0.5817,0.2452,0.1298,0.3125
3,Artisanat/Production,Homme,231,3380.0,3456.2294,1826.3642,0.5628,0.1082,0.6450,0.2900,0.1299,0.0909,0.0693,0.6364,0.6710,0.3247,0.1082,0.3636
4,Commerce informel,Femme,537,4151.0,4097.9907,2122.8691,0.4004,0.1695,0.6704,0.5196,0.1266,0.1341,0.0670,0.5736,0.6220,0.2514,0.1695,0.3296
5,Commerce informel,Homme,489,4151.0,3951.6115,2137.5872,0.4151,0.1513,0.6667,0.4581,0.1186,0.1391,0.0838,0.6564,0.5971,0.2638,0.1513,0.3149


## 13. Score de priorité pour le ciblage microcrédit par département
> **Logique du score :** un département est prioritaire pour le ciblage si
> il cumule un potentiel élevé (taux de tontine, taux de CIN, revenu médian)
> et une demande non satisfaite (taux d'exclusion financière, taux de manque
> d'argent). On construit un score composite pondéré sur 100.


In [59]:
# ── Score de priorité (0–100) ────────────────────────────────────────────────
score = agg_dept.copy()

# Normalisation min-max de chaque composante sur [0, 1]
def norm(s):
    if s.max() == s.min():
        return pd.Series([0.5]*len(s), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

# Composantes du score
score["c_tontine"]   = norm(score["taux_tontine"])          # discipline épargne
score["c_cin"]       = norm(score["taux_cin"])              # éligibilité documentaire
score["c_revenu"]    = norm(score["revenu_median"])         # capacité économique
score["c_exclusion"] = norm(score["taux_exclusion_financiere"])  # demande non satisfaite
score["c_manque"]    = norm(score["taux_manque_argent"])    # vulnérabilité / besoin
score["c_telephone"] = norm(score["taux_telephone"])        # accès potentiel numérique

# Score pondéré (total = 100 %)
score["score_priorite"] = (
    score["c_tontine"]   * 0.20 +   # discipline d'épargne
    score["c_cin"]       * 0.20 +   # éligibilité documentaire
    score["c_revenu"]    * 0.20 +   # capacité de remboursement
    score["c_exclusion"] * 0.15 +   # marché non couvert
    score["c_manque"]    * 0.15 +   # besoin exprimé
    score["c_telephone"] * 0.10     # potentiel numérique
) * 100

score["score_priorite"] = score["score_priorite"].round(1)
score = score.sort_values("score_priorite", ascending=False).reset_index(drop=True)

print("\n  Score de priorité de ciblage microcrédit par département :")
print(f"  {'Département':<25} {'Score':>8} {'Revenu médian':>15} {'% Tontine':>10} {'% CIN':>8}")
print("  " + "─"*70)
for _, r in score.iterrows():
    print(f"  {r['Departement']:<25} {r['score_priorite']:>8.1f} "
          f"{r['revenu_median']:>15,.0f} HTG {r['taux_tontine']*100:>9.1f}% "
          f"{r['taux_cin']*100:>7.1f}%")



  Score de priorité de ciblage microcrédit par département :
  Département                  Score   Revenu médian  % Tontine    % CIN
  ──────────────────────────────────────────────────────────────────────
  Nord                          68.7           3,019 HTG      44.0%    63.4%
  Sud                           68.0           2,756 HTG      41.7%    67.8%
  Aire Métropolitaine           63.1           3,373 HTG      40.5%    69.3%
  Ouest (reste)                 59.4           2,508 HTG      41.4%    62.4%
  Nord-Est                      54.6           2,694 HTG      41.4%    61.3%
  Centre                        54.3           2,161 HTG      41.6%    63.0%
  Nippes                        53.3           2,097 HTG      41.9%    59.2%
  Artibonite                    50.6           2,431 HTG      37.6%    65.2%
  Nord-Ouest                    34.0           1,540 HTG      38.2%    55.9%
  Grand-Anse                    29.5           1,894 HTG      36.8%    58.2%


## 14. Export de tous les fichiers CSV

In [60]:
# ── Export ───────────────────────────────────────────────────────────────────
exports = {
    "agg_departement.csv"      : agg_dept,
    "agg_milieu.csv"           : agg_milieu,
    "agg_secteur.csv"          : agg_secteur,
    "agg_education.csv"        : agg_educ,
    "agg_inclusion.csv"        : agg_inclusion,
    "agg_genre.csv"            : agg_genre,
    "agg_age.csv"              : agg_age,
    "agg_dept_milieu.csv"      : agg_dept_milieu,
    "agg_secteur_milieu.csv"   : agg_secteur_milieu,
    "agg_secteur_genre.csv"    : agg_secteur_genre,
    "score_ciblage_dept.csv"   : score,
}

for fname, df_out in exports.items():
    path = f"{DIR_OUT}/{fname}"
    df_out.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"   {fname:<35} → {df_out.shape[0]} lignes × {df_out.shape[1]} colonnes")

print(f"\n {len(exports)} fichiers exportés dans '{DIR_OUT}/'")



   agg_departement.csv                 → 10 lignes × 17 colonnes
   agg_milieu.csv                      → 2 lignes × 17 colonnes
   agg_secteur.csv                     → 6 lignes × 17 colonnes
   agg_education.csv                   → 4 lignes × 17 colonnes
   agg_inclusion.csv                   → 4 lignes × 17 colonnes
   agg_genre.csv                       → 2 lignes × 17 colonnes
   agg_age.csv                         → 6 lignes × 17 colonnes
   agg_dept_milieu.csv                 → 20 lignes × 18 colonnes
   agg_secteur_milieu.csv              → 12 lignes × 18 colonnes
   agg_secteur_genre.csv               → 12 lignes × 18 colonnes
   score_ciblage_dept.csv              → 10 lignes × 24 colonnes

 11 fichiers exportés dans 'Outputs\Tables/'
